# Project 13: Heart Disease Risk Prediction
**Team No.:** 19  
**Team Members:** Shubham Kumar Pradhan; Srabani Mohanty; Uttam Biswal; Satyajeet Mohapatra  
**Proposed Hybrid Model:** FT-Transformer + Residual Tabular MLP  
**Dataset:** Cardiovascular disease dataset (cardio_train.csv)  
**Source:** https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset


## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kaggle tqdm tabulate

In [ ]:
import os, json, random, glob, subprocess, sys, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    mean_absolute_error, mean_squared_error, r2_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print("Using device:", DEVICE)

### CONFIG

In [ ]:
CONFIG = {
    "project_no": "13", "project_name": "Heart Disease Risk Prediction", "team_no": "19",
    "task_type": "classification", "kaggle_dataset_slug": "sulianova/cardiovascular-disease-dataset",
    "target_column": None, "id_columns": [], "time_column": None,
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15}, "random_seed": SEED,
    "data_raw_dir": "data/13/raw", "data_processed_dir": "data/13/processed",
    "figures_dir": "data/13/figures", "results_dir": "data/13/results", "reports_dir": "data/13/reports",
    "batch_size": 64, "epochs": 30, "patience": 5, "learning_rate": 1e-3, "ft_hidden": 64,
    "use_amp": True,
}
# CONFIG["checkpoint_path"] is the single source of truth for where the checkpoint is written
# AND read back from - every training/eval cell below must use this, never a hardcoded string.
CONFIG["checkpoint_path"] = os.path.join(CONFIG["results_dir"], "best_hybrid.pt")
CONFIG["metrics_path"] = os.path.join(CONFIG["results_dir"], "metrics.json")

# Every directory any cell in this notebook writes to (raw/processed/figures/results/reports)
# is created here, upfront, in one loop - this is the fix for the "Parent directory results
# does not exist" bug: earlier code paths wrote to bare "results/" and "figures/" instead of
# CONFIG["results_dir"] / CONFIG["figures_dir"], so os.makedirs never touched the path the
# save actually used. Every save/open/savefig call below is audited to use these CONFIG paths.
for key in ["data_raw_dir", "data_processed_dir", "figures_dir", "results_dir", "reports_dir"]:
    os.makedirs(CONFIG[key], exist_ok=True)
CONFIG

## 1. Dataset Download

In [ ]:
raw = Path(CONFIG["data_raw_dir"])
if not any(raw.rglob("*")):
    subprocess.run(["kaggle", "datasets", "download", "-d", CONFIG["kaggle_dataset_slug"],
                     "-p", str(raw), "--unzip"], check=True)
for z in raw.rglob("*.zip"):
    import zipfile
    with zipfile.ZipFile(z) as f:
        f.extractall(z.parent / z.stem)
raw_files = [p for p in raw.rglob("*") if p.is_file()]
assert raw_files, "Dataset download produced no files. Configure Kaggle credentials in Colab and rerun."
assert sum(p.stat().st_size for p in raw_files) > 1024, "Downloaded content is unexpectedly small."
print(f"Discovered {len(raw_files)} files; {sum(p.stat().st_size for p in raw_files) / 2**20:.1f} MiB")

## 2. Load Raw Data

In [ ]:
csvs = list(Path(CONFIG["data_raw_dir"]).rglob("*.csv"))
assert csvs, "No CSV found"
path = max(csvs, key=lambda p: p.stat().st_size)
df = pd.read_csv(path, sep=None, engine="python")
aliases = ["cardio", "target", "label"]
target = next((c for a in aliases for c in df.columns if c.strip().lower() == a), None)
assert target is not None, f"Target not found. Columns: {df.columns.tolist()}"
CONFIG["target_column"] = target
df = df.drop_duplicates()
df = df[df[target].notna()].reset_index(drop=True)
print(path, df.shape, target)
df.head()

In [ ]:
# [TARGET SANITY CHECK] - right after the label is finalized. Hard-fails on a degenerate
# (single-class) target; soft-warns on tiny/imbalanced data.
print(df[target].value_counts())
print(df[target].value_counts(normalize=True))
assert df[target].nunique() > 1, (
    f"DEGENERATE TARGET: only {df[target].nunique()} unique value(s) found - "
    f"{df[target].value_counts().to_dict()}. Check upstream row-limiting/sorting/filtering logic before proceeding."
)
_min_class_frac = df[target].value_counts(normalize=True).min()
if _min_class_frac < 0.01 or len(df) < 100:
    print(f"[DATA QUALITY WARNING] possible degenerate/tiny target distribution - "
          f"minority class fraction={_min_class_frac:.4f}, rows={len(df)}")

## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
print(df.info())
missing = df.isna().mean().sort_values(ascending=False)

plt.figure(figsize=(6, 4))
sns.countplot(x=df[target])
plt.title("Class balance")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_class_balance.png"), dpi=150)
plt.show()

class_balance = df[target].value_counts(normalize=True).sort_index().to_dict()
memo = f"""# Data quality memo - Project 13: Heart Disease Risk Prediction

## Dataset
- Source: {path.name}
- Rows: {len(df):,}; columns: {df.shape[1]}.
- Duplicate rows were removed before splitting.
- Maximum missing fraction: {missing.max():.3f}.
- Target column: {target}
- Class balance: {class_balance}

## Known limitations
- `cardio_train.csv` is self-reported/measured survey data; blood pressure and cholesterol
  fields can contain physiologically implausible outlier values that are not manually curated.
- Preprocessors (imputer, scaler, categorical vocab) are fit only on training data.
- Stratification preserves outcome prevalence; the held-out test set is evaluated once.
"""
Path(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md")).write_text(memo, encoding="utf-8")
print(memo)

## 4. Preprocessing & Feature Engineering\n\nFeature construction is performed after splitting; every learned imputer, scaler, encoder, and vocabulary is fit on training data only.

## 5. Train / Validation / Test Split

In [ ]:
train_df, rest = train_test_split(df, test_size=0.30, stratify=df[target], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.50, stratify=rest[target], random_state=SEED)
assert not (set(train_df.index) & set(test_df.index))
assert not (set(val_df.index) & set(test_df.index))
assert not (set(train_df.index) & set(val_df.index))

drop_cols = [target] + [c for c in df.columns if c.lower() in {"id", "patient_id"}]
feature_cols = [c for c in df.columns if c not in drop_cols]

for c in feature_cols:
    if not pd.api.types.is_numeric_dtype(train_df[c]):
        vocab = {v: i + 1 for i, v in enumerate(train_df[c].astype(str).unique())}
        for part in (train_df, val_df, test_df):
            part.loc[:, c] = part[c].astype(str).map(vocab).fillna(0)

imputer = SimpleImputer(strategy="median").fit(train_df[feature_cols])
scaler = StandardScaler().fit(imputer.transform(train_df[feature_cols]))
le = LabelEncoder().fit(train_df[target].astype(str))

def transform(part):
    X = scaler.transform(imputer.transform(part[feature_cols])).astype("float32")
    y = le.transform(part[target].astype(str)).astype("int64")
    return X, y

Xtr, ytr = transform(train_df)
Xv, yv = transform(val_df)
Xte, yte = transform(test_df)

for split_name, split_y in [("train", ytr), ("val", yv), ("test", yte)]:
    assert len(np.unique(split_y)) == len(le.classes_), f"{split_name} split lost a class"

manifest = {
    "train_rows": len(ytr), "val_rows": len(yv), "test_rows": len(yte),
    "feature_cols": feature_cols, "classes": le.classes_.tolist(),
}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
manifest

## 6. PyTorch Dataset & DataLoader

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_loader = DataLoader(TabularDataset(Xtr, ytr), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(TabularDataset(Xv, yv), batch_size=CONFIG["batch_size"])
test_loader = DataLoader(TabularDataset(Xte, yte), batch_size=CONFIG["batch_size"])

xb, yb = next(iter(train_loader))
print("features:", xb.shape, "target:", yb.shape)

## 7. Proposed Model Definition — FT-Transformer + Residual Tabular MLP

In [ ]:
class FeatureTokenizer(nn.Module):
    """Projects each scalar tabular feature into its own token embedding."""
    def __init__(self, d, h):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(d, h) * 0.02)
        self.bias = nn.Parameter(torch.zeros(d, h))
    def forward(self, x):
        return x.unsqueeze(-1) * self.weight + self.bias


class FTTransformerResidualMLP(nn.Module):
    """FT-Transformer branch over per-feature tokens, fused with a residual MLP branch
    operating directly on the raw feature vector."""
    def __init__(self, d, k, h=64):
        super().__init__()
        self.token = FeatureTokenizer(d, h)
        layer = nn.TransformerEncoderLayer(h, nhead=4, dim_feedforward=128, batch_first=True, dropout=0.1)
        self.ft = nn.TransformerEncoder(layer, num_layers=2)
        self.residual = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, h))
        self.head = nn.Sequential(nn.LayerNorm(2 * h), nn.Linear(2 * h, k))
        self.last_attn_input = None  # cached for feature-importance analysis

    def forward(self, x):
        self.last_attn_input = x
        tokens = self.ft(self.token(x)).mean(1)
        res = self.residual(x)
        return self.head(torch.cat([tokens, res], dim=1))

### Architecture Verification

In [ ]:
hybrid = FTTransformerResidualMLP(Xtr.shape[1], len(le.classes_), h=CONFIG["ft_hidden"]).to(DEVICE)
print(hybrid)
total = sum(p.numel() for p in hybrid.parameters())
trainable = sum(p.numel() for p in hybrid.parameters() if p.requires_grad)
print(f"hybrid: total={total:,} trainable={trainable:,} device={next(hybrid.parameters()).device}")

## 8. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, use_amp=False):
    model.train(optimizer is not None)
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if optimizer:
            optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp and DEVICE.type == "cuda"):
            out = model(xb)
            loss = criterion(out, yb)
        if optimizer:
            if scaler is not None and scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / max(n, 1)


def train_model(model, train_loader, val_loader, checkpoint_path, epochs, patience, lr):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=2, factor=0.5)
    use_amp = bool(CONFIG.get("use_amp", True)) and DEVICE.type == "cuda"
    try:
        amp_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    history = {"train_loss": [], "val_loss": []}
    best = float("inf")
    stale = 0
    epoch_bar = tqdm(range(epochs), desc="Training", unit="epoch")
    for epoch in epoch_bar:
        tr = run_epoch(model, train_loader, criterion, opt, amp_scaler, use_amp)
        with torch.no_grad():
            va = run_epoch(model, val_loader, criterion, use_amp=use_amp)
        history["train_loss"].append(tr)
        history["val_loss"].append(va)
        scheduler.step(va)
        epoch_bar.set_postfix(train_loss=f"{tr:.5f}", val_loss=f"{va:.5f}")
        if va < best:
            best = va
            stale = 0
            torch.save(model.state_dict(), checkpoint_path)
            # Verification step: a checkpoint save must actually land on disk. This turns a
            # silent save failure (e.g. an uncreated parent dir) into an immediate loud error
            # instead of a later confusing FileNotFoundError at load time.
            assert os.path.exists(checkpoint_path), f"Checkpoint save failed: {checkpoint_path}"
        else:
            stale += 1
            if stale >= patience:
                epoch_bar.write(f"Early stopping at epoch {epoch + 1}")
                break

    assert os.path.exists(checkpoint_path), f"No checkpoint was ever saved at {checkpoint_path}"
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
    return model, history


def predict(model, loader):
    model.eval()
    pred, true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE)).cpu()
            pred.append(torch.softmax(out, dim=1))
            true.append(yb)
    return torch.cat(pred).numpy(), torch.cat(true).numpy()


hybrid, hybrid_history = train_model(
    hybrid, train_loader, val_loader, CONFIG["checkpoint_path"],
    epochs=CONFIG["epochs"], patience=CONFIG["patience"], lr=CONFIG["learning_rate"],
)

## 9. Evaluation Metrics

In [ ]:
def evaluate_classification(probs, y):
    pred = probs.argmax(1)
    pr, re, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    out = {"accuracy": accuracy_score(y, pred), "precision_macro": pr, "recall_macro": re, "f1_macro": f1}
    if probs.shape[1] == 2:
        out["roc_auc"] = roc_auc_score(y, probs[:, 1])
        out["pr_auc"] = average_precision_score(y, probs[:, 1])
    return out

results = {}
cached = {}
for name, model in [("hybrid", hybrid)]:
    probs, y = predict(model, test_loader)
    cached[name] = (probs, probs.argmax(1), y)
    results[name] = evaluate_classification(probs, y)

with open(CONFIG["metrics_path"], "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# Reload-and-verify step: load the checkpoint back into a fresh model instance and confirm
# the reloaded model reproduces the metrics above. This specifically prevents "the checkpoint
# was never actually saved" from silently recurring - a bad/missing checkpoint would either
# fail to load here or reproduce different metrics.
verify_model = FTTransformerResidualMLP(Xtr.shape[1], len(le.classes_), h=CONFIG["ft_hidden"]).to(DEVICE)
verify_model.load_state_dict(torch.load(CONFIG["checkpoint_path"], map_location=DEVICE, weights_only=True))
verify_probs, verify_y = predict(verify_model, test_loader)
verify_metrics = evaluate_classification(verify_probs, verify_y)
for k in results["hybrid"]:
    assert abs(results["hybrid"][k] - verify_metrics[k]) < 1e-6, (
        f"Reload mismatch on {k}: trained={results['hybrid'][k]} reloaded={verify_metrics[k]}"
    )
print("Reload-and-verify PASSED:", verify_metrics)

## 10. Required Figures

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(hybrid_history["train_loss"], label="hybrid train")
plt.plot(hybrid_history["val_loss"], label="hybrid val")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.title("Training / Validation Loss")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig01_loss_curves.png"), dpi=150)
plt.show()

In [ ]:
probs, pred, y = cached["hybrid"]
cm = confusion_matrix(y, pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
if probs.shape[1] == 2:
    fpr, tpr, _ = roc_curve(y, probs[:, 1])
    precision, recall, _ = precision_recall_curve(y, probs[:, 1])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "k--")
    axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate"); axes[0].set_title("ROC Curve")
    axes[1].plot(recall, precision)
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision-Recall Curve")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=150)
    plt.show()
else:
    support = np.bincount(y, minlength=probs.shape[1])
    per_class = [(pred[y == i] == i).mean() if (y == i).any() else np.nan for i in range(probs.shape[1])]
    plt.figure(figsize=(9, 4))
    plt.bar(range(len(per_class)), per_class)
    plt.ylabel("Per-class recall")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_class_performance.png"), dpi=150)
    plt.show()

### Explainability — Gradient-Based Feature Importance

In [ ]:
# Gradient x input attribution, averaged over a test batch, as a stand-in for tabular
# feature-importance explainability.
xb, yb = next(iter(test_loader))
xb = xb.to(DEVICE).requires_grad_(True)
hybrid.eval()
hybrid.zero_grad()
out = hybrid(xb)
score = out.max(1).values.sum() if out.ndim == 2 and out.shape[1] > 1 else out.sum()
score.backward()
importance = xb.grad.detach().abs().cpu().numpy().mean(axis=0)

order = np.argsort(importance)[::-1]
plt.figure(figsize=(9, 5))
plt.bar(range(len(importance)), importance[order])
plt.xticks(range(len(importance)), [feature_cols[i] for i in order], rotation=90)
plt.ylabel("Mean |gradient x input|")
plt.title("Gradient-based feature importance")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=150)
plt.show()

### Error Analysis

In [ ]:
probs, pred, y = cached["hybrid"]
errors = (pred != y).astype(int)
n_err = int(errors.sum())
print(f"Misclassified test rows: {n_err:,} / {len(y):,} ({n_err / max(len(y), 1):.2%})")

max_conf = probs.max(1)
plt.figure(figsize=(8, 4))
sns.histplot(max_conf[errors.astype(bool)], bins=20, color="crimson", label="misclassified")
sns.histplot(max_conf[~errors.astype(bool)], bins=20, color="steelblue", alpha=0.5, label="correct")
plt.xlabel("Predicted-class confidence")
plt.title("Held-out prediction confidence: correct vs. misclassified")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=150)
plt.show()